In [ ]:
from google.colab import files
uploaded = files.upload()

import random
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 86
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# Files expected
# ------------------------------------------------------------
REQUIRED_FILES = [
    "analytical_mart_therapy_region_month.csv",
    "modeling_dataset.csv",
    "historical_modeling_dataset.csv",
    "patient_informed_modeling_dataset.csv",
    "hybrid_modeling_dataset.csv",
    "feature_provenance.csv",
    "feature_availability_audit.csv",
]

# Load outputs
# ------------------------------------------------------------
analytical_mart = pd.read_csv(
    "analytical_mart_therapy_region_month.csv",
    parse_dates=["month"]
)

modeling_dataset = pd.read_csv("modeling_dataset.csv")
historical_dataset = pd.read_csv("historical_modeling_dataset.csv")
patient_dataset = pd.read_csv("patient_informed_modeling_dataset.csv")
hybrid_dataset = pd.read_csv("hybrid_modeling_dataset.csv")

feature_provenance = pd.read_csv("feature_provenance.csv")
feature_availability_audit = pd.read_csv("feature_availability_audit.csv")

# ------------------------------------------------------------
# Standardize only metadata names.
# Feature names stay unchanged.
# ------------------------------------------------------------
COLUMN_CANDIDATES = {
    "therapy": ["therapy"],
    "region": ["region"],
    "forecast_origin_month_index": [
        "forecast_origin_month_index",
        "origin_month_index",
    ],
    "target_month_index": [
        "target_month_index",
    ],
    "horizon_months": [
        "horizon_months",
        "forecast_horizon_months",
        "forecast_horizon",
    ],
    "actual_sales_units": [
        "target_observed_sales_units",
        "actual_observed_sales_units",
        "actual_sales_units",
    ],
    "target_market_regime": [
        "target_market_regime",
        "market_regime",
    ],
    "target_supply_constrained": [
        "target_supply_constrained",
        "supply_constrained",
    ],
}

def resolve_column(df, standard_name, candidates):
    matches = [column for column in candidates if column in df.columns]

    if len(matches) == 0:
        raise KeyError(
            f"Could not find a column for '{standard_name}'. "
            f"Expected one of: {candidates}"
        )

    if len(matches) > 1:
        raise ValueError(
            f"More than one candidate found for '{standard_name}': {matches}"
        )

    return matches[0]


def standardize_metadata(df, dataset_name):
    df = df.copy()

    rename_map = {}

    # Required metadata
    for standard_name in [
        "therapy",
        "region",
        "forecast_origin_month_index",
        "target_month_index",
        "horizon_months",
        "actual_sales_units",
    ]:
        source_column = resolve_column(
            df,
            standard_name,
            COLUMN_CANDIDATES[standard_name],
        )
        rename_map[source_column] = standard_name

    # Optional evaluation-only labels
    for standard_name in [
        "target_market_regime",
        "target_supply_constrained",
    ]:
        matches = [
            column
            for column in COLUMN_CANDIDATES[standard_name]
            if column in df.columns
        ]

        if len(matches) == 1:
            rename_map[matches[0]] = standard_name

    df = df.rename(columns=rename_map)

    required_columns = [
        "therapy",
        "region",
        "forecast_origin_month_index",
        "target_month_index",
        "horizon_months",
        "actual_sales_units",
    ]

    missing_columns = [
        column for column in required_columns if column not in df.columns
    ]

    assert not missing_columns, (
        f"{dataset_name} is missing required metadata: {missing_columns}"
    )

    return df


modeling_dataset = standardize_metadata(
    modeling_dataset,
    "modeling_dataset",
)

historical_dataset = standardize_metadata(
    historical_dataset,
    "historical_dataset",
)

patient_dataset = standardize_metadata(
    patient_dataset,
    "patient_dataset",
)

hybrid_dataset = standardize_metadata(
    hybrid_dataset,
    "hybrid_dataset",
)

# ------------------------------------------------------------
# Data-contract checks
# ------------------------------------------------------------
MODEL_DATASETS = {
    "Historical": historical_dataset,
    "Patient-informed": patient_dataset,
    "Hybrid": hybrid_dataset,
}

EXPECTED_HORIZONS = {1, 3, 6, 12}
KEY_COLUMNS = [
    "therapy",
    "region",
    "forecast_origin_month_index",
    "target_month_index",
    "horizon_months",
]

contract_summary = []

for dataset_name, dataset in MODEL_DATASETS.items():

    observed_horizons = set(dataset["horizon_months"].dropna().unique())

    assert observed_horizons == EXPECTED_HORIZONS, (
        f"{dataset_name}: expected horizons {EXPECTED_HORIZONS}, "
        f"but found {observed_horizons}"
    )

    assert dataset["actual_sales_units"].notna().all(), (
        f"{dataset_name}: target sales contain missing values."
    )

    assert (dataset["actual_sales_units"] >= 0).all(), (
        f"{dataset_name}: target sales contain negative values."
    )

    assert not dataset.duplicated(KEY_COLUMNS).any(), (
        f"{dataset_name}: duplicate forecast-origin rows found."
    )

    assert (
        dataset["target_month_index"]
        == dataset["forecast_origin_month_index"] + dataset["horizon_months"]
    ).all(), (
        f"{dataset_name}: target month does not equal origin + horizon."
    )

    contract_summary.append(
        {
            "dataset": dataset_name,
            "rows": len(dataset),
            "columns": len(dataset.columns),
            "therapies": dataset["therapy"].nunique(),
            "regions": dataset["region"].nunique(),
            "horizons": sorted(observed_horizons),
            "target_min": round(dataset["actual_sales_units"].min(), 2),
            "target_max": round(dataset["actual_sales_units"].max(), 2),
        }
    )

contract_summary = pd.DataFrame(contract_summary)

# ------------------------------------------------------------
# Locked temporal-validation design
# ------------------------------------------------------------
# Three years of observed target history before the first
# validation origin. All later training sets expand over time.
MIN_INITIAL_HISTORY_MONTHS = 36

# Quarterly origins keep Colab runtime proportional.
# Event anchors ensure major market changes are evaluated.
QUARTERLY_ORIGINS = set(
    range(
        MIN_INITIAL_HISTORY_MONTHS + 1,
        int(modeling_dataset["forecast_origin_month_index"].max()) + 1,
        3,
    )
)

EVENT_ANCHOR_ORIGINS = {
    54, 55, 60,       # Therapy B East access announcement/effect
    77, 78, 84, 85,   # Therapy D announcement/launch
    102, 103,         # Therapy A persistence deterioration
    108, 109, 114,    # Epidemiology announcement/effect
    123, 124, 125,    # Therapy D East supply event
}

# These labels are strictly for evaluation and must never be features.
EVALUATION_ONLY_COLUMNS = {
    "target_market_regime",
    "target_supply_constrained",
}

for dataset_name, dataset in MODEL_DATASETS.items():
    leaked_labels = EVALUATION_ONLY_COLUMNS.intersection(dataset.columns)

    # Labels may exist in the dataframe, but we record that they will be
    # explicitly excluded when feature lists are created in Block 2.
    print(
        f"{dataset_name}: evaluation-only labels present for analysis -> "
        f"{sorted(leaked_labels)}"
    )

print("\nNotebook 3 input contract passed.")
display(contract_summary)

print("\nLocked validation design")
print(f"Initial observed history: {MIN_INITIAL_HISTORY_MONTHS} months")
print("Evaluation horizons: 1, 3, 6, 12 months")
print("Forecast origins: quarterly schedule + predefined event anchors")
print(
    "Evaluation-only labels: "
    f"{sorted(EVALUATION_ONLY_COLUMNS)}"
)

print("\nFeature provenance from Notebook 2")
display(feature_provenance)

print("\nAvailability audit from Notebook 2")
display(feature_availability_audit)

In [ ]:
# ============================================================
# NOTEBOOK 03 - BLOCK 2
# Forecasting philosophies and rolling-origin framework
# ============================================================

# This cell may install XGBoost once in a new Colab runtime.
try:
    from xgboost import XGBRegressor
except ImportError:
    !pip -q install xgboost
    from xgboost import XGBRegressor

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# ------------------------------------------------------------
# 1. Explicit model feature lists
# ------------------------------------------------------------
# Evaluation labels are intentionally absent from every list.

HISTORICAL_FEATURE_CANDIDATES = [
    "sales_at_forecast_origin",
    "sales_lag_1",
    "sales_lag_3",
    "sales_lag_6",
    "sales_lag_12",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_sd_6",
    "sales_change_vs_3_months_ago",
    "therapy_age_months",
    "calendar_month",
    "calendar_year",
]

PATIENT_FEATURE_CANDIDATES = [
    "target_access_rate_allowed",
    "target_new_metastatic_diagnoses_allowed",
    "target_eligible_new_1l_patient_pool_allowed",
    "target_competitor_overlap_pressure_allowed",
]

CATEGORICAL_FEATURES = ["therapy", "region"]


def available_features(dataset, requested_features, dataset_name):
    available = [
        feature
        for feature in requested_features
        if feature in dataset.columns
    ]

    missing = [
        feature
        for feature in requested_features
        if feature not in dataset.columns
    ]

    print(f"\n{dataset_name}")
    print(f"Available features ({len(available)}): {available}")

    if missing:
        print(f"Not present ({len(missing)}): {missing}")

    return available


historical_features = available_features(
    historical_dataset,
    HISTORICAL_FEATURE_CANDIDATES,
    "Historical-model features",
)

patient_features = available_features(
    patient_dataset,
    PATIENT_FEATURE_CANDIDATES,
    "Patient-informed features",
)

hybrid_historical_features = available_features(
    hybrid_dataset,
    HISTORICAL_FEATURE_CANDIDATES,
    "Hybrid historical features",
)

hybrid_patient_features = available_features(
    hybrid_dataset,
    PATIENT_FEATURE_CANDIDATES,
    "Hybrid patient/market features",
)

hybrid_features = hybrid_historical_features + hybrid_patient_features

assert len(historical_features) >= 6, (
    "Historical feature set is unexpectedly small. "
    "Check Notebook 2 output before continuing."
)

assert len(patient_features) == 4, (
    "One or more governed patient/market features are missing. "
    "Do not continue until this is understood."
)

assert set(EVALUATION_ONLY_COLUMNS).isdisjoint(historical_features)
assert set(EVALUATION_ONLY_COLUMNS).isdisjoint(patient_features)
assert set(EVALUATION_ONLY_COLUMNS).isdisjoint(hybrid_features)

# ------------------------------------------------------------
# 2. Model constructors
# ------------------------------------------------------------
# We fit separate models by forecast horizon.
# This avoids pretending that the relationship at 1 month is
# identical to the relationship at 12 months.

def build_preprocessor(numeric_features):
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                numeric_features,
            ),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
    )


def build_historical_xgboost(numeric_features):
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(numeric_features)),
            (
                "model",
                XGBRegressor(
                    objective="reg:squarederror",
                    n_estimators=250,
                    max_depth=3,
                    learning_rate=0.04,
                    subsample=0.85,
                    colsample_bytree=0.85,
                    reg_lambda=5.0,
                    random_state=RANDOM_SEED,
                    n_jobs=2,
                ),
            ),
        ]
    )


def build_patient_driver_model(numeric_features):
    """
    Transparent patient-informed model.

    It uses only governed patient/market inputs, plus therapy and region.
    Ridge is deliberately used instead of a complex ML model because:
    - coefficients can be inspected,
    - regularization stabilizes correlated patient drivers,
    - it is a fair contrast to historical and hybrid XGBoost.
    """
    return Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(numeric_features)),
            (
                "model",
                Ridge(alpha=5.0),
            ),
        ]
    )


def fit_predict_log_scale(model, train_df, test_df, feature_columns):
    """
    Models log1p(sales), then converts back to units.

    Why:
    - demand scale differs greatly across therapies,
    - predictions remain non-negative after conversion,
    - extreme high-volume series do not dominate model fitting.
    """
    X_train = train_df[feature_columns + CATEGORICAL_FEATURES]
    X_test = test_df[feature_columns + CATEGORICAL_FEATURES]

    y_train_log = np.log1p(train_df["actual_sales_units"])

    model.fit(X_train, y_train_log)

    prediction = np.expm1(model.predict(X_test))

    return np.clip(prediction, a_min=0, a_max=None)


# ------------------------------------------------------------
# 3. Simple baselines
# ------------------------------------------------------------
# These are not weak placeholders. They are the standard that
# extra complexity must beat to claim Forecast Value Add.

def naive_forecast(test_df):
    return np.clip(
        test_df["sales_at_forecast_origin"].to_numpy(),
        a_min=0,
        a_max=None,
    )


def seasonal_naive_forecast(test_df):
    """
    Uses the same month one year earlier where available.
    Falls back to naive only if a 12-month lag is unavailable.
    """
    seasonal = test_df["sales_lag_12"].to_numpy()
    naive = naive_forecast(test_df)

    return np.where(
        np.isfinite(seasonal),
        np.clip(seasonal, a_min=0, a_max=None),
        naive,
    )


# ------------------------------------------------------------
# 4. Rolling-origin data rules
# ------------------------------------------------------------
def get_evaluation_origins(dataset, horizon_months):
    """
    Select quarterly origins plus predeclared event anchors.

    A row can be evaluated only if:
    - its origin has enough initial history,
    - its target exists within the simulation,
    - the origin is part of the locked schedule.
    """
    horizon_data = dataset.loc[
        dataset["horizon_months"] == horizon_months
    ].copy()

    eligible_origins = set(
        horizon_data.loc[
            horizon_data["forecast_origin_month_index"]
            >= MIN_INITIAL_HISTORY_MONTHS + 1,
            "forecast_origin_month_index",
        ].unique()
    )

    scheduled_origins = QUARTERLY_ORIGINS.union(EVENT_ANCHOR_ORIGINS)

    selected_origins = sorted(
        eligible_origins.intersection(scheduled_origins)
    )

    assert selected_origins, (
        f"No valid evaluation origins found for horizon {horizon_months}."
    )

    return selected_origins


def get_train_and_test_data(dataset, horizon_months, forecast_origin):
    """
    Expanding-window rule:

    Training data may contain outcomes only through the forecast origin.
    Test data are the 16 therapy-region rows forecast from that origin.
    """
    horizon_data = dataset.loc[
        dataset["horizon_months"] == horizon_months
    ].copy()

    train_df = horizon_data.loc[
        horizon_data["target_month_index"] <= forecast_origin
    ].copy()

    test_df = horizon_data.loc[
        horizon_data["forecast_origin_month_index"] == forecast_origin
    ].copy()

    assert len(test_df) == 16, (
        f"Expected 16 therapy-region forecasts at origin {forecast_origin}, "
        f"but found {len(test_df)}."
    )

    assert train_df["target_month_index"].max() <= forecast_origin
    assert (test_df["target_month_index"] > forecast_origin).all()

    return train_df, test_df


# ------------------------------------------------------------
# 5. Validation schedule audit
# ------------------------------------------------------------
validation_schedule = []

for horizon in sorted(EXPECTED_HORIZONS):
    origins = get_evaluation_origins(
        historical_dataset,
        horizon,
    )

    event_origins_included = sorted(
        set(origins).intersection(EVENT_ANCHOR_ORIGINS)
    )

    validation_schedule.append(
        {
            "horizon_months": horizon,
            "number_of_origins": len(origins),
            "first_origin_month_index": min(origins),
            "last_origin_month_index": max(origins),
            "event_anchor_origins_included": event_origins_included,
        }
    )

validation_schedule = pd.DataFrame(validation_schedule)

print("\nLocked model comparison")
comparison_summary = pd.DataFrame(
    [
        {
            "forecasting_philosophy": "Naive",
            "allowed_information": "Sales at forecast origin only",
            "model_type": "Simple baseline",
        },
        {
            "forecasting_philosophy": "Seasonal Naive",
            "allowed_information": "Sales from the same month one year earlier",
            "model_type": "Seasonal baseline",
        },
        {
            "forecasting_philosophy": "Historical XGBoost",
            "allowed_information": "Historical demand and time-derived features",
            "model_type": "Non-linear ML",
        },
        {
            "forecasting_philosophy": "Patient-informed driver model",
            "allowed_information": "Governed patient, access, epidemiology and competition features",
            "model_type": "Transparent regularized model",
        },
        {
            "forecasting_philosophy": "Hybrid XGBoost",
            "allowed_information": "Historical demand plus governed patient/market features",
            "model_type": "Non-linear ML",
        },
    ]
)

display(comparison_summary)

print("\nRolling-origin validation schedule")
display(validation_schedule)

# ------------------------------------------------------------
# 6. One safe dry run
# ------------------------------------------------------------
# This tests the full train/test timing logic before we run all
# origins in Block 3. It is not a performance result.

DRY_RUN_HORIZON = 3
DRY_RUN_ORIGIN = get_evaluation_origins(
    historical_dataset,
    DRY_RUN_HORIZON,
)[0]

dry_train_historical, dry_test_historical = get_train_and_test_data(
    historical_dataset,
    DRY_RUN_HORIZON,
    DRY_RUN_ORIGIN,
)

dry_train_patient, dry_test_patient = get_train_and_test_data(
    patient_dataset,
    DRY_RUN_HORIZON,
    DRY_RUN_ORIGIN,
)

dry_train_hybrid, dry_test_hybrid = get_train_and_test_data(
    hybrid_dataset,
    DRY_RUN_HORIZON,
    DRY_RUN_ORIGIN,
)

dry_historical_model = build_historical_xgboost(historical_features)
dry_patient_model = build_patient_driver_model(patient_features)
dry_hybrid_model = build_historical_xgboost(hybrid_features)

dry_run_predictions = dry_test_historical[
    [
        "therapy",
        "region",
        "forecast_origin_month_index",
        "target_month_index",
        "horizon_months",
        "actual_sales_units",
    ]
].copy()

dry_run_predictions["naive_prediction"] = naive_forecast(
    dry_test_historical
)

dry_run_predictions["seasonal_naive_prediction"] = seasonal_naive_forecast(
    dry_test_historical
)

dry_run_predictions["historical_xgboost_prediction"] = fit_predict_log_scale(
    dry_historical_model,
    dry_train_historical,
    dry_test_historical,
    historical_features,
)

dry_run_predictions["patient_informed_prediction"] = fit_predict_log_scale(
    dry_patient_model,
    dry_train_patient,
    dry_test_patient,
    patient_features,
)

dry_run_predictions["hybrid_xgboost_prediction"] = fit_predict_log_scale(
    dry_hybrid_model,
    dry_train_hybrid,
    dry_test_hybrid,
    hybrid_features,
)

assert (
    dry_run_predictions.filter(like="_prediction")
    .ge(0)
    .all()
    .all()
), "At least one forecast prediction is negative."

print(
    f"\nDry run passed: horizon = {DRY_RUN_HORIZON} months, "
    f"forecast origin = month {DRY_RUN_ORIGIN}"
)
print(
    f"Training rows: {len(dry_train_historical):,} | "
    f"Test rows: {len(dry_test_historical):,}"
)

display(
    dry_run_predictions.sort_values(["therapy", "region"]).round(2)
)

In [ ]:
# ============================================================
# NOTEBOOK 03 - BLOCK 3
# Rolling-origin forecasts with anchored patient-driver logic
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 86
np.random.seed(RANDOM_SEED)

HORIZONS = sorted(EXPECTED_HORIZONS)

# ------------------------------------------------------------
# 1. Metric functions
# ------------------------------------------------------------

def mae(actual, forecast):
    actual = np.asarray(actual)
    forecast = np.asarray(forecast)

    return np.mean(np.abs(actual - forecast))


def wape(actual, forecast):
    actual = np.asarray(actual)
    forecast = np.asarray(forecast)

    denominator = np.sum(np.abs(actual))

    if denominator == 0:
        return np.nan

    return np.sum(np.abs(actual - forecast)) / denominator


def forecast_bias(actual, forecast):
    """
    Positive bias means over-forecasting.
    Negative bias means under-forecasting.
    """
    actual = np.asarray(actual)
    forecast = np.asarray(forecast)

    denominator = np.sum(actual)

    if denominator == 0:
        return np.nan

    return np.sum(forecast - actual) / denominator


def summarize_metrics(df, group_cols):
    model_cols = [
        "naive_prediction",
        "seasonal_naive_prediction",
        "historical_xgboost_prediction",
        "patient_driver_prediction",
        "hybrid_xgboost_prediction",
    ]

    rows = []

    for group_values, group_df in df.groupby(group_cols):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)

        row = dict(zip(group_cols, group_values))

        for model_col in model_cols:
            model_name = model_col.replace("_prediction", "")

            row[f"{model_name}_mae"] = mae(
                group_df["actual_sales_units"],
                group_df[model_col],
            )

            row[f"{model_name}_wape"] = wape(
                group_df["actual_sales_units"],
                group_df[model_col],
            )

            row[f"{model_name}_bias"] = forecast_bias(
                group_df["actual_sales_units"],
                group_df[model_col],
            )

        row["n_forecasts"] = len(group_df)
        row["total_actual_sales"] = group_df["actual_sales_units"].sum()

        rows.append(row)

    return pd.DataFrame(rows)


def add_fva_columns(metrics_df, baseline_model="naive"):
    """
    Forecast Value Add compares each model's WAPE against naive.

    Positive FVA = better than naive.
    Negative FVA = worse than naive.
    """

    out = metrics_df.copy()
    baseline_col = f"{baseline_model}_wape"

    comparison_models = [
        "seasonal_naive",
        "historical_xgboost",
        "patient_driver",
        "hybrid_xgboost",
    ]

    for model in comparison_models:
        model_col = f"{model}_wape"
        fva_col = f"{model}_fva_vs_{baseline_model}"

        out[fva_col] = (
            out[baseline_col] - out[model_col]
        ) / out[baseline_col]

    return out


# ------------------------------------------------------------
# 2. Alignment helpers
# ------------------------------------------------------------

ALIGNMENT_COLUMNS = [
    "therapy",
    "region",
    "forecast_origin_month_index",
    "target_month_index",
    "horizon_months",
]

def sort_forecast_rows(df):
    return df.sort_values(ALIGNMENT_COLUMNS).reset_index(drop=True)


def check_alignment(df_left, df_right, left_name, right_name):
    left_keys = sort_forecast_rows(df_left)[ALIGNMENT_COLUMNS]
    right_keys = sort_forecast_rows(df_right)[ALIGNMENT_COLUMNS]

    assert left_keys.equals(right_keys), (
        f"{left_name} and {right_name} rows are not aligned."
    )


# ------------------------------------------------------------
# 3. Governed patient-driver feature engineering
# ------------------------------------------------------------

def add_target_patient_driver(df):
    """
    Creates the target-month patient opportunity signal.

    Business interpretation:
    - patient pool = size of reachable treatment opportunity
    - access = how much of the opportunity can be reached
    - competition pressure = how much overlapping competition reduces share

    This uses governed Notebook 2 features only.
    """

    out = df.copy()

    required_columns = [
        "target_access_rate_allowed",
        "target_eligible_new_1l_patient_pool_allowed",
        "target_competitor_overlap_pressure_allowed",
    ]

    missing_columns = [
        col for col in required_columns if col not in out.columns
    ]

    assert not missing_columns, (
        f"Missing required patient-driver columns: {missing_columns}"
    )

    access = out["target_access_rate_allowed"].fillna(0)

    patient_pool = out[
        "target_eligible_new_1l_patient_pool_allowed"
    ].fillna(0)

    competitor_pressure = out[
        "target_competitor_overlap_pressure_allowed"
    ].fillna(0)

    competition_adjustment = 1 - 0.35 * competitor_pressure
    competition_adjustment = competition_adjustment.clip(
        lower=0.15,
        upper=1.0,
    )

    out["target_competition_adjustment"] = competition_adjustment

    out["target_patient_driver_index"] = (
        patient_pool
        * access
        * competition_adjustment
    )

    return out

def build_origin_driver_lookup(source_df):
    """
    Builds an origin-month driver lookup.

    For a forecast made at month tau, we need:
    - the patient-driver state at tau
    - the patient-driver state at tau+h

    The dataset is horizon-level, so a row's target_month_index represents
    the month being described.

    """

    source = add_target_patient_driver(source_df).copy()

    source = source[
        [
            "therapy",
            "region",
            "forecast_origin_month_index",
            "target_month_index",
            "target_access_rate_allowed",
            "target_competitor_overlap_pressure_allowed",
            "target_patient_driver_index",
        ]
    ].copy()

    source = source.rename(
        columns={
            "forecast_origin_month_index": "view_origin_month_index",
            "target_month_index": "origin_month_index",
            "target_access_rate_allowed": "origin_access_rate",
            "target_competitor_overlap_pressure_allowed": "origin_competitor_overlap_pressure",
            "target_patient_driver_index": "origin_patient_driver_index",
        }
    )

    # Keep only views that would have been available by that origin month.
    source = source.loc[
        source["view_origin_month_index"] <= source["origin_month_index"]
    ].copy()

    source = source.sort_values(
        [
            "therapy",
            "region",
            "origin_month_index",
            "view_origin_month_index",
        ]
    )

    latest_view = (
        source
        .groupby(
            ["therapy", "region", "origin_month_index"],
            as_index=False,
        )
        .tail(1)
        .copy()
    )

    latest_view = latest_view.rename(
        columns={
            "origin_month_index": "forecast_origin_month_index",
        }
    )

    latest_view = latest_view[
        [
            "therapy",
            "region",
            "forecast_origin_month_index",
            "origin_access_rate",
            "origin_competitor_overlap_pressure",
            "origin_patient_driver_index",
        ]
    ].drop_duplicates(
        subset=[
            "therapy",
            "region",
            "forecast_origin_month_index",
        ]
    )

    assert not latest_view.columns.duplicated().any(), (
        "Origin driver lookup has duplicate column names."
    )

    return latest_view

ORIGIN_DRIVER_LOOKUP = build_origin_driver_lookup(hybrid_dataset)

print("Origin driver lookup:", ORIGIN_DRIVER_LOOKUP.shape)
display(ORIGIN_DRIVER_LOOKUP.head())


def add_origin_and_growth_features(df):
    """
    Adds:
    - target patient-driver index
    - origin patient-driver index
    - driver growth ratio from origin to target
    - access and competition changes

    These are allowed because target features already passed Notebook 2
    Forecast Information Set rules.
    """

    out = add_target_patient_driver(df)

    out = out.merge(
        ORIGIN_DRIVER_LOOKUP,
        on=["therapy", "region", "forecast_origin_month_index"],
        how="left",
        validate="many_to_one",
    )

    positive_driver_values = out.loc[
        out["target_patient_driver_index"] > 0,
        "target_patient_driver_index",
    ]

    if len(positive_driver_values) > 0:
        driver_floor = max(1.0, positive_driver_values.median() * 0.05)
    else:
        driver_floor = 1.0

    out["origin_patient_driver_index"] = (
        out["origin_patient_driver_index"].fillna(0)
    )

    out["driver_growth_ratio"] = (
        (out["target_patient_driver_index"] + driver_floor)
        / (out["origin_patient_driver_index"] + driver_floor)
    )

    out["driver_growth_ratio"] = out["driver_growth_ratio"].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    out["driver_growth_ratio"] = (
        out["driver_growth_ratio"]
        .fillna(1.0)
        .clip(lower=0.0, upper=4.0)
    )

    out["access_change_vs_origin"] = (
        out["target_access_rate_allowed"].fillna(0)
        - out["origin_access_rate"].fillna(0)
    )

    out["competitor_pressure_change_vs_origin"] = (
        out["target_competitor_overlap_pressure_allowed"].fillna(0)
        - out["origin_competitor_overlap_pressure"].fillna(0)
    )

    return out


# ------------------------------------------------------------
# 4. Enrichment helpers
# ------------------------------------------------------------

def enrich_with_reference_columns(base_df, hybrid_ref, historical_ref):
    """
    Adds columns needed for the patient-driver model.

    Patient files may not contain sales_at_forecast_origin or all governance
    flags. We recover them from aligned historical/hybrid datasets.
    """

    out = base_df.copy()

    ref_cols_from_hybrid = [
        "target_access_available_flag",
        "target_competitor_overlap_available_flag",
    ]

    ref_cols_from_historical = [
        "sales_at_forecast_origin",
    ]

    available_hybrid_cols = [
        col for col in ref_cols_from_hybrid
        if col in hybrid_ref.columns and col not in out.columns
    ]

    if available_hybrid_cols:
        hybrid_lookup = hybrid_ref[
            ALIGNMENT_COLUMNS + available_hybrid_cols
        ].drop_duplicates()

        out = out.merge(
            hybrid_lookup,
            on=ALIGNMENT_COLUMNS,
            how="left",
            validate="one_to_one",
        )

    available_historical_cols = [
        col for col in ref_cols_from_historical
        if col in historical_ref.columns and col not in out.columns
    ]

    if available_historical_cols:
        historical_lookup = historical_ref[
            ALIGNMENT_COLUMNS + available_historical_cols
        ].drop_duplicates()

        out = out.merge(
            historical_lookup,
            on=ALIGNMENT_COLUMNS,
            how="left",
            validate="one_to_one",
        )

    return out


# ------------------------------------------------------------
# 5. Anchored patient-driver model
# ------------------------------------------------------------

def estimate_driver_conversion(train_df):
    """
    Estimates sales per patient-driver unit from training data only.

    This provides a fallback for launch products where current sales are zero.
    """

    train = train_df.loc[
        (train_df["target_patient_driver_index"] > 0)
        & (train_df["actual_sales_units"] > 0)
    ].copy()

    if len(train) == 0:
        return None, None, 0

    recent_cutoff = train["target_month_index"].max() - 24

    recent_train = train.loc[
        train["target_month_index"] >= recent_cutoff
    ].copy()

    if len(recent_train) >= 20:
        train = recent_train

    therapy_region_ratio = (
        train
        .groupby(["therapy", "region"], as_index=False)
        .agg(
            actual_sales_sum=("actual_sales_units", "sum"),
            driver_sum=("target_patient_driver_index", "sum"),
        )
    )

    therapy_region_ratio["therapy_region_conversion"] = np.where(
        therapy_region_ratio["driver_sum"] > 0,
        therapy_region_ratio["actual_sales_sum"]
        / therapy_region_ratio["driver_sum"],
        np.nan,
    )

    therapy_region_ratio = therapy_region_ratio[
        ["therapy", "region", "therapy_region_conversion"]
    ]

    therapy_ratio = (
        train
        .groupby("therapy", as_index=False)
        .agg(
            actual_sales_sum=("actual_sales_units", "sum"),
            driver_sum=("target_patient_driver_index", "sum"),
        )
    )

    therapy_ratio["therapy_conversion"] = np.where(
        therapy_ratio["driver_sum"] > 0,
        therapy_ratio["actual_sales_sum"] / therapy_ratio["driver_sum"],
        np.nan,
    )

    therapy_ratio = therapy_ratio[
        ["therapy", "therapy_conversion"]
    ]

    total_driver = train["target_patient_driver_index"].sum()

    if total_driver > 0:
        global_ratio = train["actual_sales_units"].sum() / total_driver
    else:
        global_ratio = 0

    return therapy_region_ratio, therapy_ratio, global_ratio


def apply_driver_conversion(df, therapy_region_ratio, therapy_ratio, global_ratio):
    """
    Driver-only forecast:
    target patient-driver index × historical sales-per-driver conversion.
    """

    if therapy_region_ratio is None:
        return np.zeros(len(df))

    scored = df.merge(
        therapy_region_ratio,
        on=["therapy", "region"],
        how="left",
    )

    scored = scored.merge(
        therapy_ratio,
        on="therapy",
        how="left",
    )

    scored["final_conversion"] = (
        scored["therapy_region_conversion"]
        .fillna(scored["therapy_conversion"])
        .fillna(global_ratio)
    )

    pred = scored["target_patient_driver_index"] * scored["final_conversion"]

    return np.clip(pred.to_numpy(), a_min=0, a_max=None)


def choose_patient_driver_alpha(train_df):
    """
    Chooses how strongly patient-driver growth should adjust current sales.

    alpha = 0 means exactly naive.
    alpha = 1 means full patient-driver growth adjustment.

    We select alpha using training rows only.
    """

    candidates = [0.0, 0.25, 0.50, 0.75, 1.00, 1.25]

    usable = train_df.loc[
        (train_df["sales_at_forecast_origin"].fillna(0) > 0)
        & (train_df["origin_patient_driver_index"].fillna(0) > 0)
        & (train_df["target_patient_driver_index"].fillna(0) > 0)
    ].copy()

    if len(usable) < 30:
        return 0.75

    best_alpha = 0.0
    best_error = np.inf

    for alpha in candidates:
        pred = (
            usable["sales_at_forecast_origin"].to_numpy()
            * np.power(
                usable["driver_growth_ratio"].to_numpy(),
                alpha,
            )
        )

        error = wape(
            usable["actual_sales_units"],
            np.clip(pred, a_min=0, a_max=None),
        )

        if error < best_error:
            best_error = error
            best_alpha = alpha

    return best_alpha


def patient_driver_forecast(train_df, test_df):
    """
    Final patient-informed comparator.

    Main idea:
    - current sales anchor approximates active treated patient stock
    - patient/access/competition driver adjusts the forecast forward
    - driver-only fallback handles launch products with no current sales

    This is stronger than a generic patient-feature regression.
    """

    train = add_origin_and_growth_features(train_df)
    test = add_origin_and_growth_features(test_df)

    required_column = "sales_at_forecast_origin"

    assert required_column in train.columns, (
        "Patient-driver model needs sales_at_forecast_origin for calibration."
    )

    assert required_column in test.columns, (
        "Patient-driver model needs sales_at_forecast_origin for prediction."
    )

    alpha = choose_patient_driver_alpha(train)

    therapy_region_ratio, therapy_ratio, global_ratio = estimate_driver_conversion(
        train
    )

    driver_only_pred = apply_driver_conversion(
        test,
        therapy_region_ratio,
        therapy_ratio,
        global_ratio,
    )

    anchored_pred = (
        test["sales_at_forecast_origin"].fillna(0).to_numpy()
        * np.power(
            test["driver_growth_ratio"].fillna(1.0).to_numpy(),
            alpha,
        )
    )

    use_anchor = (
        (test["sales_at_forecast_origin"].fillna(0).to_numpy() > 0)
        & (test["origin_patient_driver_index"].fillna(0).to_numpy() > 0)
    )

    # Active-stock demand usually dominates short-term pharma demand.
    # The driver-only component keeps explicit patient-flow logic in the model.
    ACTIVE_STOCK_WEIGHT = 0.85

    blended_pred = (
        ACTIVE_STOCK_WEIGHT * anchored_pred
        + (1 - ACTIVE_STOCK_WEIGHT) * driver_only_pred
    )

    final_pred = np.where(
        use_anchor,
        blended_pred,
        driver_only_pred,
    )

    # Known zero access means zero sales.
    if "target_access_available_flag" in test.columns:
        known_zero_access = (
            (test["target_access_available_flag"] == 1)
            & (test["target_access_rate_allowed"].fillna(0) <= 0)
        ).to_numpy()
    else:
        known_zero_access = (
            test["target_access_rate_allowed"].fillna(0) <= 0
        ).to_numpy()

    final_pred = np.where(known_zero_access, 0, final_pred)

    return np.clip(final_pred, a_min=0, a_max=None), alpha


# ------------------------------------------------------------
# 6. Access gate for hybrid model
# ------------------------------------------------------------

def apply_known_access_gate(predictions_df):
    """
    If target access is known and zero, hybrid forecast should be zero.

    This is a business rule using allowed information, not future sales.
    """

    out = predictions_df.copy()

    if "target_access_rate_allowed" not in out.columns:
        return out

    if "target_access_available_flag" in out.columns:
        known_zero_access = (
            (out["target_access_available_flag"] == 1)
            & (out["target_access_rate_allowed"].fillna(0) <= 0)
        )
    else:
        known_zero_access = (
            out["target_access_rate_allowed"].fillna(0) <= 0
        )

    if "hybrid_xgboost_prediction" in out.columns:
        out.loc[
            known_zero_access,
            "hybrid_xgboost_prediction",
        ] = 0

    return out


# ------------------------------------------------------------
# 7. Hybrid feature expansion
# ------------------------------------------------------------

ENGINEERED_PATIENT_FEATURES = [
    "target_patient_driver_index",
    "origin_patient_driver_index",
    "driver_growth_ratio",
    "access_change_vs_origin",
    "competitor_pressure_change_vs_origin",
]

# ------------------------------------------------------------
# 8. Run full rolling-origin validation
# ------------------------------------------------------------

all_forecasts = []
alpha_audit_rows = []

for horizon in HORIZONS:
    origins = get_evaluation_origins(
        historical_dataset,
        horizon,
    )

    print(f"Running horizon {horizon} months | origins: {len(origins)}")

    for origin in origins:

        train_historical, test_historical = get_train_and_test_data(
            historical_dataset,
            horizon,
            origin,
        )

        train_patient, test_patient = get_train_and_test_data(
            patient_dataset,
            horizon,
            origin,
        )

        train_hybrid, test_hybrid = get_train_and_test_data(
            hybrid_dataset,
            horizon,
            origin,
        )

        check_alignment(
            test_historical,
            test_patient,
            "Historical test",
            "Patient-informed test",
        )

        check_alignment(
            test_historical,
            test_hybrid,
            "Historical test",
            "Hybrid test",
        )

        check_alignment(
            train_historical,
            train_patient,
            "Historical train",
            "Patient-informed train",
        )

        check_alignment(
            train_historical,
            train_hybrid,
            "Historical train",
            "Hybrid train",
        )

        test_historical = sort_forecast_rows(test_historical)
        test_patient = sort_forecast_rows(test_patient)
        test_hybrid = sort_forecast_rows(test_hybrid)

        train_historical = sort_forecast_rows(train_historical)
        train_patient = sort_forecast_rows(train_patient)
        train_hybrid = sort_forecast_rows(train_hybrid)

        train_patient = enrich_with_reference_columns(
            train_patient,
            train_hybrid,
            train_historical,
        )

        test_patient = enrich_with_reference_columns(
            test_patient,
            test_hybrid,
            test_historical,
        )

        # Add governed engineered features to hybrid model.
        train_hybrid_enhanced = add_origin_and_growth_features(train_hybrid)
        test_hybrid_enhanced = add_origin_and_growth_features(test_hybrid)

        enhanced_hybrid_features = list(
            dict.fromkeys(hybrid_features + ENGINEERED_PATIENT_FEATURES)
        )

        enhanced_hybrid_features = [
            feature
            for feature in enhanced_hybrid_features
            if feature in train_hybrid_enhanced.columns
        ]

        # Baselines
        naive_pred = naive_forecast(test_historical)

        seasonal_naive_pred = seasonal_naive_forecast(test_historical)

        # Historical XGBoost
        historical_model = build_historical_xgboost(historical_features)

        historical_pred = fit_predict_log_scale(
            historical_model,
            train_historical,
            test_historical,
            historical_features,
        )

        # Patient-informed anchored driver model
        patient_driver_pred, selected_alpha = patient_driver_forecast(
            train_patient,
            test_patient,
        )

        alpha_audit_rows.append(
            {
                "horizon_months": horizon,
                "forecast_origin_month_index": origin,
                "selected_alpha": selected_alpha,
            }
        )

        # Hybrid XGBoost with governed patient-driver features
        hybrid_model = build_historical_xgboost(enhanced_hybrid_features)

        hybrid_pred = fit_predict_log_scale(
            hybrid_model,
            train_hybrid_enhanced,
            test_hybrid_enhanced,
            enhanced_hybrid_features,
        )

        fold_result = test_historical[
            [
                "therapy",
                "region",
                "forecast_origin_month_index",
                "target_month_index",
                "horizon_months",
                "actual_sales_units",
                "target_market_regime",
                "target_supply_constrained",
            ]
        ].copy()

        for col in [
            "target_access_rate_allowed",
            "target_access_available_flag",
            "target_new_metastatic_diagnoses_allowed",
            "target_eligible_new_1l_patient_pool_allowed",
            "target_competitor_overlap_pressure_allowed",
            "target_competitor_overlap_available_flag",
        ]:
            if col in test_hybrid.columns:
                fold_result[col] = test_hybrid[col].values

        for col in ENGINEERED_PATIENT_FEATURES:
            if col in test_hybrid_enhanced.columns:
                fold_result[col] = test_hybrid_enhanced[col].values

        fold_result["naive_prediction"] = naive_pred
        fold_result["seasonal_naive_prediction"] = seasonal_naive_pred
        fold_result["historical_xgboost_prediction"] = historical_pred
        fold_result["patient_driver_prediction"] = patient_driver_pred
        fold_result["hybrid_xgboost_prediction"] = hybrid_pred

        fold_result = apply_known_access_gate(fold_result)

        all_forecasts.append(fold_result)

forecast_results = pd.concat(all_forecasts, ignore_index=True)
patient_alpha_audit = pd.DataFrame(alpha_audit_rows)

prediction_cols = [
    "naive_prediction",
    "seasonal_naive_prediction",
    "historical_xgboost_prediction",
    "patient_driver_prediction",
    "hybrid_xgboost_prediction",
]

for col in prediction_cols:
    assert forecast_results[col].notna().all(), (
        f"Missing predictions in {col}"
    )

    assert (forecast_results[col] >= 0).all(), (
        f"Negative predictions in {col}"
    )

print("\nRolling-origin forecasting complete.")
print("Forecast result shape:", forecast_results.shape)

display(
    forecast_results[
        [
            "therapy",
            "region",
            "forecast_origin_month_index",
            "target_month_index",
            "horizon_months",
            "actual_sales_units",
            "naive_prediction",
            "seasonal_naive_prediction",
            "historical_xgboost_prediction",
            "patient_driver_prediction",
            "hybrid_xgboost_prediction",
            "target_market_regime",
        ]
    ].head(12).round(2)
)


# ------------------------------------------------------------
# 9. Metrics
# ------------------------------------------------------------

metrics_by_horizon = summarize_metrics(
    forecast_results,
    group_cols=["horizon_months"],
)

metrics_by_horizon = add_fva_columns(metrics_by_horizon)

print("\nPerformance by forecast horizon:")
display(metrics_by_horizon.round(4))


metrics_by_horizon_regime = summarize_metrics(
    forecast_results,
    group_cols=[
        "horizon_months",
        "target_market_regime",
    ],
)

metrics_by_horizon_regime = add_fva_columns(metrics_by_horizon_regime)

print("\nPerformance by horizon and market regime:")
display(metrics_by_horizon_regime.round(4))


compact_rows = []

for _, row in metrics_by_horizon.iterrows():
    horizon = row["horizon_months"]

    for model in [
        "naive",
        "seasonal_naive",
        "historical_xgboost",
        "patient_driver",
        "hybrid_xgboost",
    ]:
        compact_rows.append(
            {
                "horizon_months": horizon,
                "model": model,
                "WAPE": row[f"{model}_wape"],
                "MAE": row[f"{model}_mae"],
                "Bias": row[f"{model}_bias"],
            }
        )

compact_metrics = pd.DataFrame(compact_rows)

print("\nCompact model comparison:")
display(compact_metrics.round(4))

print("\nPatient-driver alpha audit:")
display(
    patient_alpha_audit
    .groupby("horizon_months", as_index=False)
    .agg(
        mean_alpha=("selected_alpha", "mean"),
        min_alpha=("selected_alpha", "min"),
        max_alpha=("selected_alpha", "max"),
    )
    .round(3)
)


# ------------------------------------------------------------
# 10. Visual
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

for model in compact_metrics["model"].unique():
    model_df = compact_metrics[
        compact_metrics["model"] == model
    ]

    plt.plot(
        model_df["horizon_months"],
        model_df["WAPE"],
        marker="o",
        label=model,
    )

plt.title("Forecast Accuracy by Horizon")
plt.xlabel("Forecast Horizon")
plt.ylabel("WAPE")
plt.xticks(HORIZONS)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


# ------------------------------------------------------------
# 11. Save outputs
# ------------------------------------------------------------

forecast_results.to_csv("forecast_results.csv", index=False)
metrics_by_horizon.to_csv("metrics_by_horizon.csv", index=False)
metrics_by_horizon_regime.to_csv(
    "metrics_by_horizon_regime.csv",
    index=False,
)
compact_metrics.to_csv("compact_model_comparison.csv", index=False)
patient_alpha_audit.to_csv("patient_driver_alpha_audit.csv", index=False)

print("\nNotebook 3 Block 3 outputs saved:")
print("- forecast_results.csv")
print("- metrics_by_horizon.csv")
print("- metrics_by_horizon_regime.csv")
print("- compact_model_comparison.csv")
print("- patient_driver_alpha_audit.csv")

In [ ]:
# ============================================================
# NOTEBOOK 03 - BLOCK 4
# Uncertainty and scenario intelligence
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 86
np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# 1. Scenario planning view
# ------------------------------------------------------------

REQUIRED_SCENARIO_COLUMNS = [
    "therapy",
    "region",
    "forecast_origin_month_index",
    "target_month_index",
    "horizon_months",
    "actual_sales_units",
    "patient_driver_prediction",
    "target_access_rate_allowed",
    "target_eligible_new_1l_patient_pool_allowed",
    "target_competitor_overlap_pressure_allowed",
    "target_patient_driver_index",
]

missing_columns = [
    col
    for col in REQUIRED_SCENARIO_COLUMNS
    if col not in forecast_results.columns
]

assert not missing_columns, (
    f"Scenario block is missing required columns: {missing_columns}"
)

SCENARIO_HORIZON = 12

SCENARIO_ORIGIN = int(
    forecast_results.loc[
        forecast_results["horizon_months"] == SCENARIO_HORIZON,
        "forecast_origin_month_index",
    ].max()
)

baseline_planning_rows = forecast_results.loc[
    (forecast_results["horizon_months"] == SCENARIO_HORIZON)
    & (forecast_results["forecast_origin_month_index"] == SCENARIO_ORIGIN)
].copy()

assert len(baseline_planning_rows) == 16, (
    "Expected 16 therapy-region rows for the scenario planning view."
)

print("Scenario planning view")
print(f"Forecast origin month index: {SCENARIO_ORIGIN}")
print(f"Forecast horizon: {SCENARIO_HORIZON} months")
print(
    "Target month index:",
    int(baseline_planning_rows["target_month_index"].iloc[0]),
)

display(
    baseline_planning_rows[
        [
            "therapy",
            "region",
            "patient_driver_prediction",
            "target_access_rate_allowed",
            "target_eligible_new_1l_patient_pool_allowed",
            "target_competitor_overlap_pressure_allowed",
        ]
    ].round(3)
)


# ------------------------------------------------------------
# 2. Scenario catalog
# ------------------------------------------------------------

scenario_catalog = pd.DataFrame(
    [
        {
            "scenario_name": "Base case",
            "description": "No change from governed baseline forecast.",
            "access_multiplier": 1.00,
            "patient_pool_multiplier": 1.00,
            "competitor_pressure_delta": 0.00,
            "persistence_multiplier": 1.00,
            "apply_competitor_to": "none",
            "supply_fill_rate_therapy_d_east": 1.00,
        },
        {
            "scenario_name": "Access downside",
            "description": "Market access drops by 15 percent across therapies and regions.",
            "access_multiplier": 0.85,
            "patient_pool_multiplier": 1.00,
            "competitor_pressure_delta": 0.00,
            "persistence_multiplier": 1.00,
            "apply_competitor_to": "none",
            "supply_fill_rate_therapy_d_east": 1.00,
        },
        {
            "scenario_name": "Earlier strong competitor pressure",
            "description": "Overlapping competitor pressure rises for biomarker-positive therapies A, B, and D.",
            "access_multiplier": 1.00,
            "patient_pool_multiplier": 1.00,
            "competitor_pressure_delta": 0.25,
            "persistence_multiplier": 1.00,
            "apply_competitor_to": "positive_overlap",
            "supply_fill_rate_therapy_d_east": 1.00,
        },
        {
            "scenario_name": "Epidemiology upside",
            "description": "Eligible patient pool increases by 12 percent.",
            "access_multiplier": 1.00,
            "patient_pool_multiplier": 1.12,
            "competitor_pressure_delta": 0.00,
            "persistence_multiplier": 1.00,
            "apply_competitor_to": "none",
            "supply_fill_rate_therapy_d_east": 1.00,
        },
        {
            "scenario_name": "Persistence downside",
            "description": "Average persistence worsens, reducing active treated demand by 8 percent.",
            "access_multiplier": 1.00,
            "patient_pool_multiplier": 1.00,
            "competitor_pressure_delta": 0.00,
            "persistence_multiplier": 0.92,
            "apply_competitor_to": "none",
            "supply_fill_rate_therapy_d_east": 1.00,
        },
        {
            "scenario_name": "Therapy D East supply constraint",
            "description": "Therapy D in East has a 70 percent supply fill rate.",
            "access_multiplier": 1.00,
            "patient_pool_multiplier": 1.00,
            "competitor_pressure_delta": 0.00,
            "persistence_multiplier": 1.00,
            "apply_competitor_to": "none",
            "supply_fill_rate_therapy_d_east": 0.70,
        },
        {
            "scenario_name": "Combined downside",
            "description": "Access drops, competitor pressure rises, persistence worsens, and Therapy D East supply is constrained.",
            "access_multiplier": 0.85,
            "patient_pool_multiplier": 1.00,
            "competitor_pressure_delta": 0.25,
            "persistence_multiplier": 0.92,
            "apply_competitor_to": "positive_overlap",
            "supply_fill_rate_therapy_d_east": 0.70,
        },
    ]
)

print("\nScenario catalog")
display(scenario_catalog)


# ------------------------------------------------------------
# 3. Scenario engine
# ------------------------------------------------------------

POSITIVE_OVERLAP_THERAPIES = ["Therapy A", "Therapy B", "Therapy D"]


def calculate_patient_driver_index(patient_pool, access, competitor_pressure):
    """
    Converts patient opportunity, access, and competition into a demand driver.
    """

    competition_adjustment = (
        1 - 0.35 * competitor_pressure
    ).clip(lower=0.15, upper=1.0)

    return patient_pool * access * competition_adjustment


def apply_scenario(rows, scenario):
    """
    Numerical scenario engine.

    Scenario calculations are owned by this function.
    A future RAG/LLM layer may explain the outputs, but it should not
    calculate the official scenario values.
    """

    out = rows.copy()

    baseline_access = out["target_access_rate_allowed"].fillna(0)
    baseline_pool = out[
        "target_eligible_new_1l_patient_pool_allowed"
    ].fillna(0)
    baseline_pressure = out[
        "target_competitor_overlap_pressure_allowed"
    ].fillna(0).clip(lower=0)

    baseline_driver = calculate_patient_driver_index(
        patient_pool=baseline_pool,
        access=baseline_access,
        competitor_pressure=baseline_pressure,
    )

    out["scenario_access_rate"] = (
        baseline_access * scenario["access_multiplier"]
    ).clip(lower=0, upper=1)

    out["scenario_patient_pool"] = (
        baseline_pool * scenario["patient_pool_multiplier"]
    ).clip(lower=0)

    out["scenario_competitor_pressure"] = baseline_pressure.copy()

    if scenario["apply_competitor_to"] == "positive_overlap":
        affected_rows = out["therapy"].isin(POSITIVE_OVERLAP_THERAPIES)

        out.loc[affected_rows, "scenario_competitor_pressure"] = (
            out.loc[affected_rows, "scenario_competitor_pressure"]
            + scenario["competitor_pressure_delta"]
        )

    out["scenario_competitor_pressure"] = (
        out["scenario_competitor_pressure"].clip(lower=0)
    )

    scenario_driver = calculate_patient_driver_index(
        patient_pool=out["scenario_patient_pool"],
        access=out["scenario_access_rate"],
        competitor_pressure=out["scenario_competitor_pressure"],
    )

    driver_floor = max(
        1.0,
        baseline_driver.loc[baseline_driver > 0].median() * 0.05,
    )

    out["scenario_driver_ratio"] = (
        (scenario_driver + driver_floor)
        / (baseline_driver + driver_floor)
    )

    out["scenario_driver_ratio"] = (
        out["scenario_driver_ratio"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(1.0)
        .clip(lower=0, upper=4)
    )

    out["baseline_forecast_units"] = out["patient_driver_prediction"]

    out["scenario_forecast_units_before_supply"] = (
        out["baseline_forecast_units"]
        * out["scenario_driver_ratio"]
        * scenario["persistence_multiplier"]
    )

    out["scenario_supply_fill_rate"] = 1.0

    therapy_d_east = (
        (out["therapy"] == "Therapy D")
        & (out["region"] == "East")
    )

    out.loc[
        therapy_d_east,
        "scenario_supply_fill_rate",
    ] = scenario["supply_fill_rate_therapy_d_east"]

    out["scenario_forecast_units"] = (
        out["scenario_forecast_units_before_supply"]
        * out["scenario_supply_fill_rate"]
    )

    if scenario["scenario_name"] == "Base case":
        out["scenario_forecast_units"] = out["baseline_forecast_units"]

    out["scenario_name"] = scenario["scenario_name"]
    out["scenario_description"] = scenario["description"]

    out["absolute_change_units"] = (
        out["scenario_forecast_units"]
        - out["baseline_forecast_units"]
    )

    out["percent_change"] = np.where(
        out["baseline_forecast_units"] > 0,
        out["absolute_change_units"] / out["baseline_forecast_units"],
        np.nan,
    )

    return out


scenario_row_outputs = []

for _, scenario in scenario_catalog.iterrows():
    scenario_row_outputs.append(
        apply_scenario(
            baseline_planning_rows,
            scenario,
        )
    )

scenario_row_outputs = pd.concat(
    scenario_row_outputs,
    ignore_index=True,
)

print("\nScenario row outputs created")
display(
    scenario_row_outputs[
        [
            "scenario_name",
            "therapy",
            "region",
            "baseline_forecast_units",
            "scenario_forecast_units",
            "absolute_change_units",
            "percent_change",
        ]
    ].round(3).head(24)
)


# ------------------------------------------------------------
# 4. Scenario summaries
# ------------------------------------------------------------

scenario_summary = (
    scenario_row_outputs
    .groupby("scenario_name", as_index=False)
    .agg(
        baseline_forecast_units=("baseline_forecast_units", "sum"),
        scenario_forecast_units=("scenario_forecast_units", "sum"),
        absolute_change_units=("absolute_change_units", "sum"),
    )
)

scenario_summary["percent_change"] = (
    scenario_summary["absolute_change_units"]
    / scenario_summary["baseline_forecast_units"]
)

scenario_summary = scenario_summary.merge(
    scenario_catalog[["scenario_name", "description"]],
    on="scenario_name",
    how="left",
)

therapy_scenario_summary = (
    scenario_row_outputs
    .groupby(["scenario_name", "therapy"], as_index=False)
    .agg(
        baseline_forecast_units=("baseline_forecast_units", "sum"),
        scenario_forecast_units=("scenario_forecast_units", "sum"),
        absolute_change_units=("absolute_change_units", "sum"),
    )
)

therapy_scenario_summary["percent_change"] = (
    therapy_scenario_summary["absolute_change_units"]
    / therapy_scenario_summary["baseline_forecast_units"]
)

print("\nOverall scenario summary")
display(scenario_summary.round(4))

print("\nTherapy-level scenario summary")
display(therapy_scenario_summary.round(4))


# ------------------------------------------------------------
# 5. Monte Carlo uncertainty
# ------------------------------------------------------------

def beta_parameters_from_mean(mean, concentration):
    """
    Converts a probability mean into Beta distribution parameters.
    """

    mean = np.clip(mean, 0.001, 0.999)

    alpha = mean * concentration
    beta = (1 - mean) * concentration

    return alpha, beta


def simulate_scenario_uncertainty(rows, scenario, n_simulations=1000):
    """
    Lightweight Monte Carlo uncertainty.

    Uncertain inputs:
    - access rate
    - eligible patient pool
    - competitor pressure
    - persistence multiplier

    Output:
    P10, P50, P90 forecasted total units.
    """

    base = rows.copy()

    baseline_access = base["target_access_rate_allowed"].fillna(0)
    baseline_pool = base[
        "target_eligible_new_1l_patient_pool_allowed"
    ].fillna(0)
    baseline_pressure = base[
        "target_competitor_overlap_pressure_allowed"
    ].fillna(0).clip(lower=0)

    baseline_driver = calculate_patient_driver_index(
        patient_pool=baseline_pool,
        access=baseline_access,
        competitor_pressure=baseline_pressure,
    )

    scenario_access = (
        baseline_access * scenario["access_multiplier"]
    ).clip(lower=0, upper=1)

    scenario_pool = (
        baseline_pool * scenario["patient_pool_multiplier"]
    ).clip(lower=0)

    scenario_pressure = baseline_pressure.copy()

    if scenario["apply_competitor_to"] == "positive_overlap":
        affected_rows = base["therapy"].isin(POSITIVE_OVERLAP_THERAPIES)

        scenario_pressure.loc[affected_rows] = (
            scenario_pressure.loc[affected_rows]
            + scenario["competitor_pressure_delta"]
        )

    scenario_pressure = scenario_pressure.clip(lower=0)

    driver_floor = max(
        1.0,
        baseline_driver.loc[baseline_driver > 0].median() * 0.05,
    )

    simulated_totals = []

    for _ in range(n_simulations):
        access_draws = []

        for value in scenario_access:
            if value <= 0:
                access_draws.append(0)
            elif value >= 1:
                access_draws.append(1)
            else:
                a, b = beta_parameters_from_mean(
                    value,
                    concentration=120,
                )
                access_draws.append(np.random.beta(a, b))

        access_draws = np.array(access_draws)

        patient_pool_draws = np.random.lognormal(
            mean=np.log(np.maximum(scenario_pool.to_numpy(), 1)),
            sigma=0.06,
        )

        pressure_draws = np.random.normal(
            loc=scenario_pressure.to_numpy(),
            scale=0.04,
        )

        pressure_draws = np.clip(pressure_draws, 0, None)

        simulated_driver = calculate_patient_driver_index(
            patient_pool=patient_pool_draws,
            access=access_draws,
            competitor_pressure=pd.Series(pressure_draws),
        ).to_numpy()

        simulated_driver_ratio = (
            (simulated_driver + driver_floor)
            / (baseline_driver.to_numpy() + driver_floor)
        )

        simulated_driver_ratio = np.clip(
            np.nan_to_num(simulated_driver_ratio, nan=1.0),
            0,
            4,
        )

        persistence_draw = np.random.triangular(
            left=max(0.70, scenario["persistence_multiplier"] - 0.05),
            mode=scenario["persistence_multiplier"],
            right=min(1.20, scenario["persistence_multiplier"] + 0.05),
        )

        simulated_forecast = (
            base["patient_driver_prediction"].to_numpy()
            * simulated_driver_ratio
            * persistence_draw
        )

        supply_fill = np.ones(len(base))

        therapy_d_east = (
            (base["therapy"] == "Therapy D")
            & (base["region"] == "East")
        ).to_numpy()

        supply_fill[therapy_d_east] = scenario[
            "supply_fill_rate_therapy_d_east"
        ]

        simulated_forecast = simulated_forecast * supply_fill

        if scenario["scenario_name"] == "Base case":
            simulated_forecast = (
                base["patient_driver_prediction"].to_numpy()
                * persistence_draw
            )

        simulated_totals.append(simulated_forecast.sum())

    simulated_totals = np.array(simulated_totals)

    return {
        "scenario_name": scenario["scenario_name"],
        "p10": np.percentile(simulated_totals, 10),
        "p50": np.percentile(simulated_totals, 50),
        "p90": np.percentile(simulated_totals, 90),
        "mean": simulated_totals.mean(),
        "sd": simulated_totals.std(),
    }


uncertainty_rows = []

for _, scenario in scenario_catalog.iterrows():
    uncertainty_rows.append(
        simulate_scenario_uncertainty(
            baseline_planning_rows,
            scenario,
            n_simulations=1000,
        )
    )

scenario_uncertainty_summary = pd.DataFrame(uncertainty_rows)

scenario_summary_with_uncertainty = scenario_summary.merge(
    scenario_uncertainty_summary,
    on="scenario_name",
    how="left",
)

print("\nScenario summary with uncertainty")
display(scenario_summary_with_uncertainty.round(3))


# ------------------------------------------------------------
# 6. Validation checks
# ------------------------------------------------------------

base_total = scenario_summary.loc[
    scenario_summary["scenario_name"] == "Base case",
    "scenario_forecast_units",
].iloc[0]

base_baseline_total = scenario_summary.loc[
    scenario_summary["scenario_name"] == "Base case",
    "baseline_forecast_units",
].iloc[0]

access_downside_total = scenario_summary.loc[
    scenario_summary["scenario_name"] == "Access downside",
    "scenario_forecast_units",
].iloc[0]

combined_downside_total = scenario_summary.loc[
    scenario_summary["scenario_name"] == "Combined downside",
    "scenario_forecast_units",
].iloc[0]

epidemiology_upside_total = scenario_summary.loc[
    scenario_summary["scenario_name"] == "Epidemiology upside",
    "scenario_forecast_units",
].iloc[0]

persistence_downside_total = scenario_summary.loc[
    scenario_summary["scenario_name"] == "Persistence downside",
    "scenario_forecast_units",
].iloc[0]

supply_constraint_total = scenario_summary.loc[
    scenario_summary["scenario_name"] == "Therapy D East supply constraint",
    "scenario_forecast_units",
].iloc[0]

assert np.isclose(base_total, base_baseline_total), (
    "Base case must equal the baseline forecast."
)

assert access_downside_total < base_total, (
    "Access downside should reduce forecasted demand."
)

assert combined_downside_total < access_downside_total, (
    "Combined downside should be lower than access downside alone."
)

assert epidemiology_upside_total > base_total, (
    "Epidemiology upside should increase forecasted demand."
)

assert persistence_downside_total < base_total, (
    "Persistence downside should reduce forecasted demand."
)

assert supply_constraint_total < base_total, (
    "Therapy D East supply constraint should reduce total observed sales."
)

competitor_therapy_summary = therapy_scenario_summary.loc[
    therapy_scenario_summary["scenario_name"]
    == "Earlier strong competitor pressure"
].copy()

for therapy in POSITIVE_OVERLAP_THERAPIES:
    therapy_change = competitor_therapy_summary.loc[
        competitor_therapy_summary["therapy"] == therapy,
        "absolute_change_units",
    ].iloc[0]

    assert therapy_change <= 0, (
        f"{therapy} should not increase under stronger overlapping competition."
    )

therapy_c_change = competitor_therapy_summary.loc[
    competitor_therapy_summary["therapy"] == "Therapy C",
    "percent_change",
].iloc[0]

assert abs(therapy_c_change) < 0.01, (
    "Therapy C should have little or no impact from positive-overlap competition."
)

supply_rows = scenario_row_outputs.loc[
    scenario_row_outputs["scenario_name"]
    == "Therapy D East supply constraint"
].copy()

therapy_d_east_change = supply_rows.loc[
    (supply_rows["therapy"] == "Therapy D")
    & (supply_rows["region"] == "East"),
    "absolute_change_units",
].iloc[0]

assert therapy_d_east_change < 0, (
    "Therapy D East should fall under the supply constraint."
)

non_therapy_d_east_change = supply_rows.loc[
    ~(
        (supply_rows["therapy"] == "Therapy D")
        & (supply_rows["region"] == "East")
    ),
    "absolute_change_units",
].abs().sum()

assert non_therapy_d_east_change < 1e-6, (
    "Supply constraint scenario should only affect Therapy D East."
)

print("\nScenario validation checks passed.")


# ------------------------------------------------------------
# 7. Visuals
# ------------------------------------------------------------

plot_df = scenario_summary.loc[
    scenario_summary["scenario_name"] != "Base case"
].copy()

plt.figure(figsize=(10, 5))

plt.barh(
    plot_df["scenario_name"],
    plot_df["absolute_change_units"],
)

plt.axvline(0, color="black", linewidth=1)

plt.title("Scenario Impact vs Baseline Forecast")
plt.xlabel("Change in forecasted units")
plt.ylabel("Scenario")
plt.grid(True, axis="x", alpha=0.3)
plt.show()


plt.figure(figsize=(10, 5))

plt.errorbar(
    scenario_summary_with_uncertainty["scenario_name"],
    scenario_summary_with_uncertainty["p50"],
    yerr=[
        scenario_summary_with_uncertainty["p50"]
        - scenario_summary_with_uncertainty["p10"],
        scenario_summary_with_uncertainty["p90"]
        - scenario_summary_with_uncertainty["p50"],
    ],
    fmt="o",
    capsize=5,
)

plt.xticks(rotation=45, ha="right")
plt.title("Scenario Forecast Uncertainty")
plt.ylabel("Forecasted units")
plt.grid(True, axis="y", alpha=0.3)
plt.show()


# ------------------------------------------------------------
# 8. Outputs for RAG
# ------------------------------------------------------------

scenario_catalog.to_csv("scenario_catalog.csv", index=False)
scenario_row_outputs.to_csv("scenario_row_outputs.csv", index=False)
scenario_summary.to_csv("scenario_summary.csv", index=False)
therapy_scenario_summary.to_csv(
    "therapy_scenario_summary.csv",
    index=False,
)
scenario_uncertainty_summary.to_csv(
    "scenario_uncertainty_summary.csv",
    index=False,
)
scenario_summary_with_uncertainty.to_csv(
    "scenario_summary_with_uncertainty.csv",
    index=False,
)

print("\nNotebook 3 Block 4 outputs saved:")
print("- scenario_catalog.csv")
print("- scenario_row_outputs.csv")
print("- scenario_summary.csv")
print("- therapy_scenario_summary.csv")
print("- scenario_uncertainty_summary.csv")
print("- scenario_summary_with_uncertainty.csv")

In [ ]:
# ============================================================
# NOTEBOOK 03 - BLOCK 5
# Senior review gate and final evidence summary
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Best model by horizon
# ------------------------------------------------------------

best_model_by_horizon = (
    compact_metrics
    .sort_values(["horizon_months", "WAPE"])
    .groupby("horizon_months", as_index=False)
    .first()
)

print("Best model by forecast horizon")
display(best_model_by_horizon.round(4))


# ------------------------------------------------------------
# 2. Forecast Value Add summary
# ------------------------------------------------------------

fva_summary = metrics_by_horizon[
    [
        "horizon_months",
        "historical_xgboost_fva_vs_naive",
        "patient_driver_fva_vs_naive",
        "hybrid_xgboost_fva_vs_naive",
    ]
].copy()

print("\nForecast Value Add vs naive")
display(fva_summary.round(4))


# ------------------------------------------------------------
# 3. Regime-level signals worth discussing
# ------------------------------------------------------------

regime_signal_summary = metrics_by_horizon_regime[
    [
        "horizon_months",
        "target_market_regime",
        "naive_wape",
        "historical_xgboost_wape",
        "patient_driver_wape",
        "hybrid_xgboost_wape",
        "historical_xgboost_fva_vs_naive",
        "patient_driver_fva_vs_naive",
        "hybrid_xgboost_fva_vs_naive",
        "n_forecasts",
    ]
].copy()

# Keep cases where any non-naive method improved over naive.
regime_positive_fva = regime_signal_summary.loc[
    (
        (regime_signal_summary["historical_xgboost_fva_vs_naive"] > 0)
        | (regime_signal_summary["patient_driver_fva_vs_naive"] > 0)
        | (regime_signal_summary["hybrid_xgboost_fva_vs_naive"] > 0)
    )
].copy()

print("\nRegime windows where a non-naive method added value")
display(regime_positive_fva.round(4))


# ------------------------------------------------------------
# 4. Scenario conclusions
# ------------------------------------------------------------

scenario_conclusion_table = scenario_summary_with_uncertainty[
    [
        "scenario_name",
        "scenario_forecast_units",
        "absolute_change_units",
        "percent_change",
        "p10",
        "p50",
        "p90",
    ]
].copy()

scenario_conclusion_table = scenario_conclusion_table.sort_values(
    "scenario_forecast_units"
)

print("\nScenario conclusion table")
display(scenario_conclusion_table.round(3))


# ------------------------------------------------------------
# 5. Claim discipline
# ------------------------------------------------------------

claim_discipline = pd.DataFrame(
    [
        {
            "claim_type": "Supported",
            "claim": (
                "Recent observed demand was the strongest overall forecast "
                "benchmark in this synthetic oncology market."
            ),
        },
        {
            "claim_type": "Supported",
            "claim": (
                "Patient and market signals were useful for event-regime "
                "analysis and scenario planning."
            ),
        },
        {
            "claim_type": "Supported",
            "claim": (
                "Forecast Information Set governance prevented ordinary "
                "forecast models from using unavailable future information."
            ),
        },
        {
            "claim_type": "Supported",
            "claim": (
                "Scenario outputs were traceable from access, epidemiology, "
                "competition, persistence, and supply assumptions to demand."
            ),
        },
        {
            "claim_type": "Do not claim",
            "claim": (
                "Hybrid XGBoost beat all simpler methods across all horizons."
            ),
        },
        {
            "claim_type": "Do not claim",
            "claim": (
                "The synthetic case study proves real-world oncology demand behavior."
            ),
        },
        {
            "claim_type": "Do not claim",
            "claim": (
                "The model is clinically validated or ready for production deployment."
            ),
        },
    ]
)

print("\nClaim discipline")
display(claim_discipline)


# ------------------------------------------------------------
# 6. Senior review gate
# ------------------------------------------------------------

senior_review_gate = pd.DataFrame(
    [
        {
            "review_area": "Research question",
            "assessment": "PASS",
            "comment": (
                "The notebook evaluates when historical demand, patient logic, "
                "and hybrid information help forecasting."
            ),
        },
        {
            "review_area": "Temporal validation",
            "assessment": "PASS",
            "comment": (
                "Rolling-origin validation is used across 1, 3, 6, and 12-month horizons."
            ),
        },
        {
            "review_area": "Leakage control",
            "assessment": "PASS",
            "comment": (
                "Forecast Information Set rules from Notebook 2 are preserved."
            ),
        },
        {
            "review_area": "Baselines",
            "assessment": "PASS",
            "comment": (
                "Naive and seasonal naive baselines are included, and naive is not ignored."
            ),
        },
        {
            "review_area": "Patient-informed comparator",
            "assessment": "PASS",
            "comment": (
                "The final patient-driver model anchors to current demand and adjusts "
                "using governed patient, access, and competition signals."
            ),
        },
        {
            "review_area": "Hybrid model",
            "assessment": "PASS WITH LIMITATION",
            "comment": (
                "Hybrid XGBoost adds value in selected regimes but not overall versus naive."
            ),
        },
        {
            "review_area": "Scenario engine",
            "assessment": "PASS",
            "comment": (
                "Scenario outputs move in clinically and commercially logical directions."
            ),
        },
        {
            "review_area": "Uncertainty",
            "assessment": "PASS",
            "comment": (
                "Monte Carlo intervals provide practical P10/P50/P90 planning ranges."
            ),
        },
        {
            "review_area": "External validity",
            "assessment": "LIMITATION",
            "comment": (
                "The analysis is a synthetic case study and would need real data validation."
            ),
        },
        {
            "review_area": "Client POC value",
            "assessment": "PASS",
            "comment": (
                "The notebook supports planning conversations around access, competition, "
                "persistence, epidemiology, supply, and uncertainty."
            ),
        },
    ]
)

print("\nSenior review gate")
display(senior_review_gate)


# ------------------------------------------------------------
# 7. Conclusion
# ------------------------------------------------------------

notebook_3_conclusion = pd.DataFrame(
    [
        {
            "question": "When is history enough?",
            "answer": (
                "History was strongest for routine forecasting because observed sales "
                "already captured active treated patient stock."
            ),
        },
        {
            "question": "When does patient/market information help?",
            "answer": (
                "It helped most as a scenario and event-intelligence layer, especially "
                "when assumptions about persistence, access, competition, epidemiology, "
                "or supply changed."
            ),
        },
        {
            "question": "What should a planner do?",
            "answer": (
                "Use simple demand anchors for stable short-term forecasting, but use "
                "patient-informed scenario analysis when future market assumptions change."
            ),
        },
        {
            "question": "What is the main limitation?",
            "answer": (
                "The case is synthetic and simplified; real-world deployment would require "
                "validated epidemiology, access, claims, sales, and treatment-flow data."
            ),
        },
    ]
)

display(notebook_3_conclusion)


# ------------------------------------------------------------
# 8. Final outputs
# ------------------------------------------------------------

best_model_by_horizon.to_csv("best_model_by_horizon.csv", index=False)
fva_summary.to_csv("fva_summary.csv", index=False)
regime_signal_summary.to_csv("regime_signal_summary.csv", index=False)
regime_positive_fva.to_csv("regime_positive_fva.csv", index=False)
scenario_conclusion_table.to_csv("scenario_conclusion_table.csv", index=False)
claim_discipline.to_csv("claim_discipline.csv", index=False)
senior_review_gate.to_csv("notebook_3_senior_review_gate.csv", index=False)
notebook_3_conclusion.to_csv("notebook_3_conclusion.csv", index=False)


## Validation addition: therapy balance and origin-level uncertainty

This section checks whether the portfolio conclusion changes when each therapy receives equal weight and quantifies uncertainty in model-versus-naive comparisons. The bootstrap resamples complete forecast origins, keeping the 16 therapy-region rows within each origin together. This respects the evaluation design and avoids treating correlated rows from the same planning date as independent.

**Interpretation boundary:** these intervals measure backtest stability across forecast origins. They are not clinical confidence intervals and they do not validate the synthetic market externally.


In [ ]:
# ============================================================
# NOTEBOOK 3 - VALIDATION APPENDIX
# Therapy-level metrics, macro metrics, regime FVA and
# forecast-origin block-bootstrap intervals
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


RANDOM_SEED = 86
BOOTSTRAP_REPLICATIONS = 2000

MODEL_COLUMNS = {
    "naive": "naive_prediction",
    "seasonal_naive": "seasonal_naive_prediction",
    "historical_xgboost": "historical_xgboost_prediction",
    "patient_driver": "patient_driver_prediction",
    "hybrid_xgboost": "hybrid_xgboost_prediction",
}


def _find_forecast_results():
    candidates = [
        Path("forecast_results.csv"),
        Path("data/outputs/forecast_results.csv"),
        Path("../data/outputs/forecast_results.csv"),
        Path("/content/forecast_results.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate)
    raise FileNotFoundError(
        "forecast_results.csv was not found. Upload it or run the earlier Notebook 3 blocks first."
    )


def _output_directory():
    repo_output = Path("data/outputs")
    if repo_output.parent.exists():
        repo_output.mkdir(parents=True, exist_ok=True)
        return repo_output
    return Path(".")


def _wape(actual, prediction):
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    denominator = np.abs(actual).sum()
    return np.nan if denominator == 0 else np.abs(actual - prediction).sum() / denominator


def _mae(actual, prediction):
    return np.abs(np.asarray(actual, dtype=float) - np.asarray(prediction, dtype=float)).mean()


def _bias(actual, prediction):
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    denominator = actual.sum()
    return np.nan if denominator == 0 else (prediction - actual).sum() / denominator


def validate_forecast_results(df):
    required = {
        "therapy", "region", "forecast_origin_month_index", "horizon_months",
        "actual_sales_units", "target_market_regime", *MODEL_COLUMNS.values(),
    }
    missing = sorted(required.difference(df.columns))
    if missing:
        raise KeyError(f"forecast_results is missing required columns: {missing}")
    if df[list(MODEL_COLUMNS.values()) + ["actual_sales_units"]].isna().any().any():
        raise ValueError("Forecast results contain missing actuals or predictions.")
    if (df[list(MODEL_COLUMNS.values())] < 0).any().any():
        raise ValueError("Forecast results contain negative predictions.")


def calculate_therapy_metrics(df):
    rows = []
    for (horizon, therapy), group in df.groupby(["horizon_months", "therapy"], sort=True):
        for model, prediction_col in MODEL_COLUMNS.items():
            rows.append({
                "horizon_months": int(horizon),
                "therapy": therapy,
                "model": model,
                "WAPE": _wape(group["actual_sales_units"], group[prediction_col]),
                "MAE": _mae(group["actual_sales_units"], group[prediction_col]),
                "Bias": _bias(group["actual_sales_units"], group[prediction_col]),
                "n_forecasts": len(group),
                "total_actual_sales": group["actual_sales_units"].sum(),
            })
    return pd.DataFrame(rows)


def calculate_macro_metrics(therapy_metrics):
    return (
        therapy_metrics
        .groupby(["horizon_months", "model"], as_index=False)
        .agg(
            macro_WAPE=("WAPE", "mean"),
            macro_MAE=("MAE", "mean"),
            macro_Bias=("Bias", "mean"),
            therapies=("therapy", "nunique"),
            n_forecasts=("n_forecasts", "sum"),
        )
        .sort_values(["horizon_months", "macro_WAPE"])
        .reset_index(drop=True)
    )


def calculate_regime_positive_fva(df):
    rows = []
    for (horizon, regime), group in df.groupby(
        ["horizon_months", "target_market_regime"], sort=True
    ):
        row = {
            "horizon_months": int(horizon),
            "target_market_regime": regime,
            "n_forecasts": len(group),
            "total_actual_sales": group["actual_sales_units"].sum(),
        }
        for model, prediction_col in MODEL_COLUMNS.items():
            row[f"{model}_wape"] = _wape(group["actual_sales_units"], group[prediction_col])
        naive_wape = row["naive_wape"]
        for model in MODEL_COLUMNS:
            if model == "naive":
                continue
            model_wape = row[f"{model}_wape"]
            row[f"{model}_fva_vs_naive"] = (
                np.nan if naive_wape == 0 else (naive_wape - model_wape) / naive_wape
            )
        rows.append(row)

    regime = pd.DataFrame(rows)
    fva_columns = [
        f"{model}_fva_vs_naive" for model in MODEL_COLUMNS if model != "naive"
    ]
    return regime.loc[regime[fva_columns].gt(0).any(axis=1)].reset_index(drop=True)


def _origin_aggregates(horizon_df, prediction_col):
    work = horizon_df[[
        "forecast_origin_month_index", "actual_sales_units",
        "naive_prediction", prediction_col,
    ]].copy()
    work["naive_abs_error"] = (
        work["actual_sales_units"] - work["naive_prediction"]
    ).abs()
    work["model_abs_error"] = (
        work["actual_sales_units"] - work[prediction_col]
    ).abs()
    return (
        work.groupby("forecast_origin_month_index", as_index=False)
        .agg(
            actual_sum=("actual_sales_units", "sum"),
            naive_abs_error=("naive_abs_error", "sum"),
            model_abs_error=("model_abs_error", "sum"),
        )
        .sort_values("forecast_origin_month_index")
    )


def calculate_origin_bootstrap(df, replications=BOOTSTRAP_REPLICATIONS, seed=RANDOM_SEED):
    """Paired block bootstrap that resamples complete forecast origins.

    Positive delta_WAPE and positive FVA mean improvement over naive.
    All therapy-region rows from a sampled origin remain together.
    """
    rng = np.random.default_rng(seed)
    rows = []

    for horizon in sorted(df["horizon_months"].unique()):
        horizon_df = df.loc[df["horizon_months"] == horizon]
        for model, prediction_col in MODEL_COLUMNS.items():
            if model == "naive":
                continue
            blocks = _origin_aggregates(horizon_df, prediction_col)
            actual = blocks["actual_sum"].to_numpy(float)
            naive_error = blocks["naive_abs_error"].to_numpy(float)
            model_error = blocks["model_abs_error"].to_numpy(float)
            n_origins = len(blocks)

            naive_point = naive_error.sum() / actual.sum()
            model_point = model_error.sum() / actual.sum()
            delta_point = naive_point - model_point
            fva_point = delta_point / naive_point if naive_point else np.nan

            sample_indices = rng.integers(0, n_origins, size=(replications, n_origins))
            actual_boot = actual[sample_indices].sum(axis=1)
            naive_boot = naive_error[sample_indices].sum(axis=1) / actual_boot
            model_boot = model_error[sample_indices].sum(axis=1) / actual_boot
            delta_boot = naive_boot - model_boot
            fva_boot = np.divide(
                delta_boot,
                naive_boot,
                out=np.full(replications, np.nan),
                where=naive_boot != 0,
            )

            rows.append({
                "horizon_months": int(horizon),
                "model": model,
                "n_origins": n_origins,
                "bootstrap_replications": replications,
                "naive_WAPE": naive_point,
                "model_WAPE": model_point,
                "delta_WAPE_vs_naive": delta_point,
                "delta_WAPE_ci_lower": np.nanpercentile(delta_boot, 2.5),
                "delta_WAPE_ci_median": np.nanpercentile(delta_boot, 50),
                "delta_WAPE_ci_upper": np.nanpercentile(delta_boot, 97.5),
                "FVA_vs_naive": fva_point,
                "FVA_ci_lower": np.nanpercentile(fva_boot, 2.5),
                "FVA_ci_median": np.nanpercentile(fva_boot, 50),
                "FVA_ci_upper": np.nanpercentile(fva_boot, 97.5),
                "probability_model_beats_naive": np.mean(delta_boot > 0),
            })
    return pd.DataFrame(rows)


def run_forecast_validation(forecast_df=None):
    if forecast_df is None:
        forecast_df = globals().get("forecast_results")
    if forecast_df is None:
        forecast_df = _find_forecast_results()

    validate_forecast_results(forecast_df)
    therapy_metrics = calculate_therapy_metrics(forecast_df)
    macro_metrics = calculate_macro_metrics(therapy_metrics)
    regime_positive_fva = calculate_regime_positive_fva(forecast_df)
    bootstrap_intervals = calculate_origin_bootstrap(forecast_df)

    output_dir = _output_directory()
    therapy_metrics.to_csv(output_dir / "forecast_metrics_by_therapy_horizon.csv", index=False)
    macro_metrics.to_csv(output_dir / "forecast_macro_metrics_by_horizon.csv", index=False)
    bootstrap_intervals.to_csv(output_dir / "forecast_model_bootstrap_intervals.csv", index=False)
    regime_positive_fva.to_csv(output_dir / "regime_positive_fva.csv", index=False)

    print("Therapy-level metrics")
    display(therapy_metrics.round(4)) if "display" in globals() else print(therapy_metrics.round(4))
    print("\nMacro metrics")
    display(macro_metrics.round(4)) if "display" in globals() else print(macro_metrics.round(4))
    print("\nForecast-origin bootstrap intervals")
    display(bootstrap_intervals.round(4)) if "display" in globals() else print(bootstrap_intervals.round(4))
    print("\nValidation outputs saved to:", output_dir.resolve())

    return therapy_metrics, macro_metrics, bootstrap_intervals, regime_positive_fva


if __name__ == "__main__":
    run_forecast_validation()
